In [2]:
import pymysql
import pandas as pd
import re

# ── 1. Preprocessing ──────────────────────
df = pd.read_csv("../../datafile/[DArt-B 5기] Global_Supermarket.csv")

def clean_text(v):
    if pd.isna(v): return ""
    return re.sub(r"\r\n|\r|\n", " ", str(v)).strip()

# 텍스트 컬럼 줄바꿈 제거
text_cols = [
    "customer_id", "customer_name", "customer_segment",
    "order_id", "order_city", "oreder_region",
    "product_id", "product_name",
    "category", "sub_category",
    "market_country", "market_area", "market_city",
    "ship_mode"
]
for col in text_cols:
    df[col] = df[col].map(clean_text)

# 날짜 컬럼 변환 (문자열 → datetime.date)
df["order_date"] = pd.to_datetime(df["order_date"]).dt.date
df["ship_date"]  = pd.to_datetime(df["ship_date"]).dt.date

# NaN → None (SQL NULL로 처리)
df = df.where(pd.notnull(df), None)

print(f"전처리 완료: {len(df):,}행")


# ── 2. DB Connection ────────────────────────────────
conn = pymysql.connect(
    host="127.0.0.1", port=3306,
    user="root", password="2bhanlee!!",
    db="global_supermarket_db",          # 사용할 DB명으로 변경하세요
    charset="utf8mb4"
    # local_infile 옵션 불필요 (executemany 방식)
)
cur = conn.cursor()


# ── 3. Table Generation ────────────────────────────
cur.execute("""
CREATE TABLE IF NOT EXISTS global_supermarket (
    row_id           INT,
    customer_id      VARCHAR(20),
    customer_name    VARCHAR(100),
    customer_segment VARCHAR(50),
    order_id         VARCHAR(30),
    order_city       VARCHAR(100),
    order_region     VARCHAR(100),   -- 원본 오타(oreder_region) → 정규화
    order_date       DATE,
    order_year       SMALLINT,
    order_weeknum    TINYINT,
    quantity         INT,
    sales            INT,
    product_id       VARCHAR(30),
    product_name     VARCHAR(300),
    profit           DECIMAL(10,4),
    discount         INT,
    category         VARCHAR(50),
    sub_category     VARCHAR(50),
    market_country   VARCHAR(100),
    market_area      VARCHAR(10),
    market_city      VARCHAR(100),
    ship_date        DATE,
    ship_mode        VARCHAR(50),
    shipping_cost    DECIMAL(10,4)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")


# ── 4. INSERT via executemany (1,000행 배치) ─────────────────
sql = """
    INSERT INTO global_supermarket (
        customer_id, customer_name, customer_segment,
        order_id, order_city, order_region,
        order_date, order_year, order_weeknum,
        quantity, sales,
        product_id, product_name,
        profit, discount,
        category, sub_category,
        market_country, market_area, market_city,
        ship_date, ship_mode, shipping_cost,
        row_id
    ) VALUES (%s,%s,%s, %s,%s,%s, %s,%s,%s, %s,%s, %s,%s, %s,%s, %s,%s, %s,%s,%s, %s,%s,%s, %s)
"""

# CSV 컬럼 순서 (oreder_region: 원본 오타 그대로 참조)
cols = [
    "customer_id", "customer_name", "customer_segment",
    "order_id", "order_city", "oreder_region",
    "order_date", "order_year", "order_weeknum",
    "quantity", "sales",
    "product_id", "product_name",
    "profit", "discount",
    "category", "sub_category",
    "market_country", "market_area", "market_city",
    "ship_date", "ship_mode", "shipping_cost",
    "row_id"
]

BATCH_SIZE = 1000
total      = len(df)

for start in range(0, total, BATCH_SIZE):
    batch = df[cols].iloc[start : start + BATCH_SIZE]
    rows  = [tuple(r) for r in batch.itertuples(index=False, name=None)]
    cur.executemany(sql, rows)
    conn.commit()
    print(f"  {min(start + BATCH_SIZE, total):,} / {total:,} 행 삽입", end="\r")

print()


# ── 5. Confirmation ──────────────────────────────────
cur.execute("SELECT COUNT(*) FROM global_supermarket")
print(f"Insert Complete: {cur.fetchone()[0]:,} Rows")

conn.close()

전처리 완료: 51,290행
  51,290 / 51,290 행 삽입
Insert Complete: 51,290 Rows
